In [1]:
import os
import pandas as pd

from io import BytesIO
from dotenv import load_dotenv
from sqlalchemy import create_engine
from urllib.parse import quote_plus
from azure.storage.filedatalake import DataLakeServiceClient


# 1. CARREGAR CREDENCIAIS

load_dotenv()

servidor = os.getenv("SQL_SERVER")
banco = os.getenv("SQL_DATABASE")
connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")


# 2. CONECTAR AO SQL SERVER

parametros = quote_plus(
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={servidor};"
    f"DATABASE={banco};"
    f"Trusted_Connection=yes;"
)

engine = create_engine(
    f"mssql+pyodbc:///?odbc_connect={parametros}"
)

print("Conexão com SQL Server realizada com sucesso!")


# 3. CONECTAR AO AZURE DATA LAKE

datalake_service = DataLakeServiceClient.from_connection_string(
    connection_string
)

file_system = datalake_service.get_file_system_client(
    file_system="bronze"
)

diretorio = file_system.get_directory_client(
    "comercial"
)

print("Conexão com Azure Data Lake realizada com sucesso!")


# 4. DEFINIR TABELAS

tabelas = [
    "clientes",
    "pedidos",
    "itens_pedido",
    "pagamentos",
    "produtos"
]


# 5. EXTRAIR E ENVIAR PARA BRONZE

for tabela in tabelas:

    print(f"\nProcessando: {tabela}")

    # Extrair dados do SQL Server
    df = pd.read_sql(
        f"SELECT * FROM {tabela}",
        engine
    )

    print(f"Registros extraídos: {len(df)}")

    # Converter DataFrame para Parquet em memória
    buffer = BytesIO()

    df.to_parquet(
        buffer,
        index=False,
        engine="pyarrow"
    )

    # Definir arquivo no Azure
    arquivo = diretorio.get_file_client(
        f"{tabela}.parquet"
    )

    # Enviar para Bronze
    arquivo.upload_data(
        buffer.getvalue(),
        overwrite=True
    )

    print(f"{tabela}.parquet enviado para Bronze.")


# 6. FINALIZAÇÃO

engine.dispose()

print("\nIngestão concluída com sucesso!")

Conexão com SQL Server realizada com sucesso!
Conexão com Azure Data Lake realizada com sucesso!

Processando: clientes
Registros extraídos: 10000
clientes.parquet enviado para Bronze.

Processando: pedidos
Registros extraídos: 50000
pedidos.parquet enviado para Bronze.

Processando: itens_pedido
Registros extraídos: 125000
itens_pedido.parquet enviado para Bronze.

Processando: pagamentos
Registros extraídos: 53000
pagamentos.parquet enviado para Bronze.

Processando: produtos
Registros extraídos: 1000
produtos.parquet enviado para Bronze.

Ingestão concluída com sucesso!
